# 01_build_dataset.ipynb
Domain-aware dataset builder (single-receiver only).

Protocol: Train on A -> Fine-tune on B -> Predict unseen locations.


In [ ]:
from pathlib import Path
import json, re, random
import numpy as np
import pandas as pd
SEED = 42
WINDOW_ROOT = Path("dataset_windows")
OUT_DIR = Path("dataset_build_hybrid")
OUT_DIR.mkdir(parents=True, exist_ok=True)
A_TRAIN_FRAC = 0.9
B_FINETUNE_FRAC = 0.8
BUILD_B_VAL = True
TRAIN_UNLABELED_ENABLED = True


In [ ]:
def infer_receiver_domain(path: Path):
    txt = str(path).lower()
    if '/a/' in txt or '\a\' in txt: return 'A'
    if '/b/' in txt or '\b\' in txt: return 'B'
    if re.search(r'(^|[_\-])a([_\-]|$)', path.stem.lower()): return 'A'
    if re.search(r'(^|[_\-])b([_\-]|$)', path.stem.lower()): return 'B'
    return 'unknown'

def infer_anchor_label(path: Path):
    stop={'a','b','train','pred','eval','test'}
    for token in reversed(path.parts[:-1]):
        t=token.lower()
        if t in stop or t.startswith('session') or t.startswith('receiver'): continue
        return token
    return 'unknown_anchor'

def infer_session_id(path: Path):
    for token in reversed(path.parts):
        t=token.lower()
        if t.startswith('session') or t.startswith('sess'): return token
    return path.parent.name


In [ ]:
def scan_single_receiver_windows(window_root: Path):
    rows=[]
    for amp_path in sorted(window_root.rglob('amp_window_*.npy')):
        if 'ampavg' in amp_path.name: continue
        idx=int(amp_path.stem.split('_')[-1])
        p=amp_path.parent
        rows.append({
            'amp_path':str(amp_path), 'ampn_path':str(p/f'ampn_window_{idx:05d}.npy'),
            'ampavg_path':str(p/f'ampavg_window_{idx:05d}.npy'), 'pha_path':str(p/f'pha_window_{idx:05d}.npy'),
            'meta_path':str(p/f'meta_{idx:05d}.mat'), 'window_index':idx, 'source_file':str(p),
            'receiver_domain':infer_receiver_domain(amp_path), 'anchor_label':infer_anchor_label(amp_path),
            'session_id':infer_session_id(amp_path)
        })
    df=pd.DataFrame(rows)
    if df.empty: raise RuntimeError('No single-receiver windows found')
    df['sample_id']=df.apply(lambda r:f"{r.receiver_domain}|{r.session_id}|{r.anchor_label}|{r.window_index}",axis=1)
    return df
all_df=scan_single_receiver_windows(WINDOW_ROOT)
A_df=all_df[all_df.receiver_domain=='A'].reset_index(drop=True)
B_df=all_df[all_df.receiver_domain=='B'].reset_index(drop=True)
assert len(A_df)>0 and len(B_df)>0, 'Both A and B domains are required' 


In [ ]:
def split_A(df_A):
    g=df_A.groupby(['anchor_label','session_id']).size().reset_index()[['anchor_label','session_id']]
    g=g.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    cut=max(1,int(len(g)*A_TRAIN_FRAC))
    tr=set(tuple(x) for x in g.iloc[:cut].to_records(index=False))
    m=df_A.apply(lambda r:(r.anchor_label,r.session_id) in tr,axis=1)
    return df_A[m].reset_index(drop=True), df_A[~m].reset_index(drop=True)

def split_B(df_B):
    anchors=sorted(df_B.anchor_label.unique().tolist())
    random.Random(SEED).shuffle(anchors)
    cut=max(1,int(len(anchors)*B_FINETUNE_FRAC))
    fin=set(anchors[:cut]); evals=set(anchors[cut:]) or {anchors[-1]}
    B_pool=df_B[df_B.anchor_label.isin(fin)].reset_index(drop=True)
    target_eval_unseen_locations=df_B[df_B.anchor_label.isin(evals)].reset_index(drop=True)
    if BUILD_B_VAL and len(B_pool)>4:
        g=B_pool.groupby(['anchor_label','session_id']).size().reset_index()[['anchor_label','session_id']]
        g=g.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n=max(1,int(0.2*len(g))); val=set(tuple(x) for x in g.iloc[:n].to_records(index=False))
        m=B_pool.apply(lambda r:(r.anchor_label,r.session_id) in val,axis=1)
        B_val=B_pool[m].reset_index(drop=True); B_finetune=B_pool[~m].reset_index(drop=True)
    else:
        B_finetune=B_pool; B_val=pd.DataFrame(columns=B_pool.columns)
    assert not (set(B_finetune.anchor_label)&set(target_eval_unseen_locations.anchor_label))
    return B_finetune, B_val, target_eval_unseen_locations

A_train,A_val=split_A(A_df)
B_finetune,B_val,target_eval_unseen_locations=split_B(B_df)
train_unlabeled=A_train.copy() if TRAIN_UNLABELED_ENABLED else pd.DataFrame(columns=A_train.columns)
split_frames={'A_train':A_train,'A_val':A_val,'B_finetune':B_finetune,'B_val':B_val,'target_eval_unseen_locations':target_eval_unseen_locations,'train_unlabeled':train_unlabeled}


In [ ]:
def pack_df(df):
    out={}
    for c in df.columns:
        out[c]=df[c].astype(str).values if df[c].dtype==object else df[c].values
    return out
for k,v in split_frames.items(): np.savez_compressed(OUT_DIR/f'{k}.npz', **pack_df(v))
summary={'protocol':'train_A_finetune_B_predict_unseen_locations','single_receiver_only':True,'splits':{k:int(len(v)) for k,v in split_frames.items()},'notes':{'no_AB_pairing':True,'no_AB_tensor_concat':True,'target_eval_unseen_locations_excluded_from_B_finetune':True,'train_unlabeled_fallback_enabled':TRAIN_UNLABELED_ENABLED}}
(OUT_DIR/'dataset_manifest.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
